**Import Required Libraries**

In [0]:
from pyspark.sql import functions as F

**Define start and end dates**

In [0]:
start_date = "2024-01-01"
end_date   = "2024-12-31"

**Generate one row per day — airline needs daily grain (not monthly)**

In [0]:
# Generate one row per day between start and end date
df = (
    spark.sql(f"""
        SELECT explode(
            sequence(
                to_date('{start_date}'),
                to_date('{end_date}'),
                interval 1 day
            )
        ) AS flight_date
    """)
)

# Add analytics columns — same pattern as your FMCG dim_date
df = (
    df
    .withColumn("date_key",        F.date_format("flight_date", "yyyyMMdd").cast("int"))
    .withColumn("year",            F.year("flight_date"))
    .withColumn("month",           F.month("flight_date"))
    .withColumn("month_name",      F.date_format("flight_date", "MMMM"))
    .withColumn("month_short_name",F.date_format("flight_date", "MMM"))
    .withColumn("day_of_month",    F.dayofmonth("flight_date"))
    .withColumn("day_of_week",     F.dayofweek("flight_date"))     # 1=Sunday, 7=Saturday
    .withColumn("day_name",        F.date_format("flight_date", "EEEE"))
    .withColumn("is_weekend",      (F.dayofweek("flight_date").isin([1, 7])).cast("int"))
    .withColumn("quarter",         F.concat(F.lit("Q"), F.quarter("flight_date")))
    .withColumn("year_quarter",    F.concat(F.col("year"), F.lit("-Q"), F.quarter("flight_date")))
    .withColumn("year_month",      F.date_format("flight_date", "yyyy-MM"))
    .withColumn("month_start_date",F.trunc("flight_date", "MM"))
)

print(f"Total days generated: {df.count()}")
display(df.limit(10))

**Save as Delta table — airline.gold.dim_date**

In [0]:
df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("airline.gold.dim_date")

print("✅ dim_date saved → airline.gold.dim_date")

**Verify**

In [0]:
%sql
SELECT * FROM airline.gold.dim_date LIMIT 10;